In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score  # классы несбалансированы
from sklearn.metrics import average_precision_score  # положительный класс -- редкий
from sklearn.metrics import balanced_accuracy_score  # accuracy для дисбаланса
from sklearn.metrics import f1_score

In [ ]:
DATA_DIR = Path('./ml-project-adaai_VK_predict_hackaton/data')

train_path = DATA_DIR / 'train.csv'
test_path = DATA_DIR / 'test.csv'
submit_path = DATA_DIR / 'submit.csv'

for path in [train_path, test_path, submit_path]:
    print(f'{path}:', 'OK' if path.exists() else 'NOT FOUND')


In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
submit_df = pd.read_csv(submit_path)

df = train_df.copy()

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('submit shape:', submit_df.shape)


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
test_df.head()


In [ ]:
submit_df.head()


In [ ]:
uniq_values = {col: df[col].unique() for col in df.columns}

num_of_unique_values = np.array([len(df[col].unique()) for col in df.columns])

fig, ax = plt.subplots(1, 4, figsize=(14, 4), sharey=True)

num_of_uniq_filt = num_of_unique_values[num_of_unique_values < 100]
ax[0].hist(list(num_of_uniq_filt), bins=20)
ax[0].set_xlabel('Количество уникальных \nзначений < 100')
ax[0].set_ylabel('Количество признаков')
ax[0].set_title('Общее количество признаков\n в этом диапазоне: ' + str(len(num_of_uniq_filt)))

num_of_uniq_filt = num_of_unique_values[(num_of_unique_values >= 100) & (num_of_unique_values < 1000)]
ax[1].hist(list(num_of_uniq_filt), bins=20)
ax[1].set_xlabel('Количество уникальных \nзначений от 100 до 1000')
ax[1].set_ylabel('Количество признаков')
ax[1].set_title('Общее количество признаков\n в этом диапазоне: ' + str(len(num_of_uniq_filt)))

num_of_uniq_filt = num_of_unique_values[(num_of_unique_values >= 1000) & (num_of_unique_values < 10000)]
ax[2].hist(list(num_of_uniq_filt), bins=20)
ax[2].set_xlabel('Количество уникальных \nзначений от 1000 до 10000')
ax[2].set_ylabel('Количество признаков')
ax[2].set_title('Общее количество признаков\n в этом диапазоне: ' + str(len(num_of_uniq_filt)))

num_of_uniq_filt = num_of_unique_values[num_of_unique_values >= 10000]
ax[3].hist(list(num_of_uniq_filt), bins=20)
ax[3].set_xlabel('Количество уникальных \nзначений >= 10000')
ax[3].set_ylabel('Количество признаков')
ax[3].set_title('Общее количество признаков\n в этом диапазоне: ' + str(len(num_of_uniq_filt)))

plt.show()
None


In [ ]:
train_df = train_df.drop_duplicates()
train_df.dropna()

for col in train_df.columns:
    if len(train_df.loc[:, col].unique()) < 2:
        print("dropped", col)

        train_df = train_df.drop(columns=[col])


In [ ]:
train_df.shape, train_df.head()

In [ ]:
X_ = train_df.drop(columns=["index", "target"])
y_ = train_df.loc[:, "target"]

X_train, X_test, y_train, y_test = train_test_split(X_, y_, test_size=0.3, random_state=42)

X_train.shape, y_train.shape, X_test, y_test

In [ ]:
zeros_count = len(y_train[y_train==0])
ones_count = len(y_train[y_train==1])
total = len(y_train)

print(zeros_count / total)


In [ ]:
def count_metrics(y_test, y_pred):
    print("roc auc:", roc_auc_score(y_test, y_pred))
    print("average precision:", average_precision_score(y_test, y_pred))
    print("balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
    print("f1:", f1_score(y_test, y_pred))
    

### Предсказание нулем

In [ ]:
y_pred_zeros = np.zeros(y_test.shape)


In [ ]:
count_metrics(y_test, y_pred_zeros)

### Предсказание схожим с `train` распределением

In [ ]:
rng = np.random.default_rng(42)

p_one = ones_count / total
p_zero = zeros_count / total

y_pred_random = rng.choice(
    [0, 1],
    size=y_test.shape[0],
    p=[p_zero, p_one]
)

In [ ]:
count_metrics(y_test, y_pred_random)

### Logistic Regression

In [ ]:
logreg = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

logreg.fit(X_train, y_train)


In [ ]:
y_pred_logreg = logreg.predict(X_test)

In [ ]:
count_metrics(y_test, y_pred_logreg)

### Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42,
)

rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
count_metrics(y_test, y_pred_rf)